In [0]:
USE CATALOG market_risk_dev;
USE SCHEMA bronze;


In [0]:
DROP TABLE IF EXISTS market_risk_dev.bronze.market_prices;

CREATE TABLE IF NOT EXISTS market_risk_dev.bronze.market_prices 
(
  Date          DATE          COMMENT 'Trading date',
  Open          DOUBLE        COMMENT 'Opening price',
  High          DOUBLE        COMMENT 'Intraday high price',
  Low           DOUBLE        COMMENT 'Intraday low price',
  Close         DOUBLE        COMMENT 'Closing price',
  Volume        DOUBLE        COMMENT 'Number of shares or contracts traded',
  ticker        STRING        COMMENT 'Yahoo Finance ticker symbol',
  fetched_at    STRING        COMMENT 'Timestamp when data was fetched from API',
  _source_file  STRING        COMMENT 'S3 file this row was loaded from',
  _ingested_at  TIMESTAMP     COMMENT 'Timestamp when row was loaded into Bronze'
)
USING DELTA
COMMENT 'Raw OHLCV market prices from Yahoo Finance — loaded as-is, no transformations'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
COPY INTO market_risk_dev.bronze.market_prices
FROM (
  SELECT
    CAST(Date AS DATE) AS Date,
    CAST(Open AS DOUBLE) AS Open,
    CAST(High AS DOUBLE) AS High,
    CAST(Low AS DOUBLE) AS Low,
    CAST(Close AS DOUBLE) AS Close,
    CAST(Volume AS DOUBLE) AS Volume,
    CAST(ticker AS STRING)    AS ticker,
    CAST(fetched_at AS STRING) AS fetched_at,
    _metadata.file_path  AS _source_file,
    current_timestamp()  AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/prices/'
)
FILEFORMAT = CSV
PATTERN = 'year=*/month=*/day=*/*.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

In [0]:
DROP TABLE IF EXISTS market_risk_dev.bronze.positions;

CREATE TABLE IF NOT EXISTS market_risk_dev.bronze.positions (
  trade_id      STRING    COMMENT 'Unique trade identifier',
  desk          STRING    COMMENT 'Trading desk that owns the position',
  book_id       STRING    COMMENT 'Sub-desk level aggregation - a desk can have multiple books',
  trader_id     STRING    COMMENT 'Individual trader ID',  
  counterparty  STRING    COMMENT 'Counterparty name',
  asset_class   STRING    COMMENT 'Asset class: FX, Equity, Rates, Credit',
  instrument_type  STRING    COMMENT 'Distinguished product within an asset class',
  ticker        STRING    COMMENT 'Instrument ticker',
  isin          STRING    COMMENT 'International Securities Identification Number',
  cusip         STRING    COMMENT 'US CUSIP number',
  direction     STRING    COMMENT 'LONG or SHORT',
  notional      DOUBLE    COMMENT 'Notional value in local currency',
  currency      STRING    COMMENT 'Local currency of the notional',
  trade_date    DATE      COMMENT 'Date trade was booked',
  maturity_date DATE      COMMENT 'Date trade matures',
  generated_at  STRING    COMMENT 'Timestamp when synthetic data was generated',
  _source_file  STRING    COMMENT 'S3 file this row was loaded from',
  _ingested_at  TIMESTAMP COMMENT 'Timestamp when row was loaded into Bronze'
)
USING DELTA
COMMENT 'Synthetic trading book — 300 positions across 4 desks'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
);

COPY INTO market_risk_dev.bronze.positions
FROM (
  SELECT
    trade_id,
    desk,
    book_id,
    trader_id,
    counterparty,
    asset_class,
    instrument_type,
    ticker,
    isin,
    cusip,
    direction,
    CAST(notional AS DOUBLE)    AS notional,
    currency,
    CAST(trade_date AS DATE)    AS trade_date,
    CAST(maturity_date AS DATE) AS maturity_date,
    CAST(generated_at AS STRING) AS generated_at,
    _metadata.file_path         AS _source_file,
    current_timestamp()         AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/positions/'
)
FILEFORMAT = CSV
PATTERN = 'year=*/month=*/day=*/*.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
);

In [0]:
DROP TABLE IF EXISTS market_risk_dev.bronze.desk_limits;

CREATE TABLE IF NOT EXISTS market_risk_dev.bronze.desk_limits (
  desk             STRING    COMMENT 'Trading desk name',
  limit_usd        DOUBLE    COMMENT 'Risk limit in USD',
  limit_currency   STRING    COMMENT 'Limit denomination currency',
  effective_date   DATE      COMMENT 'Date limit became effective',
  review_date      DATE      COMMENT 'Date limit is next reviewed',
  approved_by      STRING    COMMENT 'Name of the limit approver',
  uploaded_by      STRING    COMMENT 'Name of the limit uploader',
  approved_date    DATE      COMMENT 'Date when the limit was approved',
  comments         STRING    COMMENT 'Comment from the approver of the limit',
  generated_at     STRING    COMMENT 'Timestamp when reference data was generated',
  _source_file     STRING    COMMENT 'S3 file this row was loaded from',
  _ingested_at     TIMESTAMP COMMENT 'Timestamp when row was loaded into Bronze'
)
USING DELTA
COMMENT 'Risk limits per trading desk in USD'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
);

COPY INTO market_risk_dev.bronze.desk_limits
FROM (
  SELECT
    desk,
    CAST(limit_usd AS DOUBLE)        AS limit_usd,
    limit_currency,
    CAST(effective_date AS DATE)     AS effective_date,
    CAST(review_date AS DATE)        AS review_date,
    CAST(generated_at AS STRING)       AS generated_at,
    _metadata.file_path              AS _source_file,
    current_timestamp()              AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/reference/'
)
FILEFORMAT = CSV
PATTERN = 'desk_limits.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'force' = 'true'
);

In [0]:

DROP TABLE IF EXISTS market_risk_dev.bronze.fx_rates;

CREATE TABLE IF NOT EXISTS market_risk_dev.bronze.fx_rates (
  currency      STRING    COMMENT 'Currency code e.g. EUR, GBP, JPY',
  rate_vs_usd   DOUBLE    COMMENT 'Exchange rate vs USD — multiply to convert to USD',
  as_of_date    DATE      COMMENT 'Date the rate is valid for',
  generated_at  STRING    COMMENT 'Timestamp when reference data was generated',
  _source_file  STRING    COMMENT 'S3 file this row was loaded from',
  _ingested_at  TIMESTAMP COMMENT 'Timestamp when row was loaded into Bronze'
)
USING DELTA
COMMENT 'FX conversion rates vs USD for reference data'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
);

COPY INTO market_risk_dev.bronze.fx_rates
FROM (
  SELECT
    currency,
    CAST(rate_vs_usd AS DOUBLE) AS rate_vs_usd,
    CAST(as_of_date AS DATE)    AS as_of_date,
    CAST(generated_at AS STRING) AS generated_at,
    _metadata.file_path         AS _source_file,
    current_timestamp()         AS _ingested_at
  FROM 's3://market-risk-dev-raw-593153124201-ap-southeast-2-an/raw/reference/'
)
FILEFORMAT = CSV
PATTERN = 'fx_rates.csv'
FORMAT_OPTIONS (
  'header'    = 'true',
  'inferSchema' = 'false'
)
COPY_OPTIONS (
  'force' = 'true'
);

In [0]:
SELECT "market_prices" AS table_name, fetched_at, count(1) AS row_count 
  FROM market_risk_dev.bronze.market_prices
  group by fetched_at
 UNION ALL
SELECT "positions", generated_at, count(1) 
  FROM market_risk_dev.bronze.positions
 GROUP BY generated_at
 UNION ALL
SELECT "desk_limits", generated_at, count(1) 
  FROM market_risk_dev.bronze.desk_limits
 GROUP BY generated_at
 UNION ALL
SELECT "fx_rates", generated_at, count(1) 
  FROM market_risk_dev.bronze.fx_rates
GROUP BY generated_at
ORDER BY 1;